# Demo AI – từ ảnh MRI đến gợi ý bệnh theo tầng

Notebook này làm 2 việc:

- **Phần A** – giải thích hai thư mục chính của repo: `pipeline/` (nghiên cứu offline trên 515 bệnh nhân) và `ai/` (xử lý từng ca mới cho web).
- **Phần B** – chạy đầu cuối như worker của BE: ảnh **DICOM (zip một lần chụp) hoặc NIfTI** → `segment()` → `analyze()` (kèm so với ngưỡng y văn), rồi vẽ lại hình [`demo_0100.png`](demo_0100.png). Chỉ dùng code có sẵn trong repo.

Chạy bằng `.venv` ở gốc repo (Python 3.12). Mặc định **không cần GPU / TotalSpineSeg**: notebook dùng mask có sẵn.

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/hieunghuu/mmis-gr8-lowbackpain"

# gốc repo = thư mục cha gần nhất có cả ai/ và pipeline/src/
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "ai").is_dir() and (p / "pipeline" / "src").is_dir()), None)
if ROOT is None and IN_COLAB:          # máy ảo Colab trống -> clone repo
    ROOT = Path.cwd() / "mmis-gr8-lowbackpain"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(ROOT)], check=True)
assert ROOT, "Không thấy gốc repo: mở notebook từ trong repo, hoặc chạy trên Colab"
print("Gốc repo:", ROOT)

PIP_C = ""   # ràng buộc cho mọi lệnh %pip trong notebook
if IN_COLAB:   # google-colab ghim cứng pandas, requests... -> cấm pip nâng các gói này (nâng là hỏng Colab)
    from importlib.metadata import requires
    pins = [r.split(";")[0].strip() for r in requires("google-colab") or [] if "==" in r]
    Path("/tmp/colab_constraints.txt").write_text("\n".join(pins))
    PIP_C = "-c /tmp/colab_constraints.txt"
%pip install -q {PIP_C} -r "{ROOT / 'pipeline' / 'requirements.txt'}" nibabel

### Cấu hình

In [ ]:
import json, os, sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk

# ROOT lấy từ ô B0
sys.path[:0] = [str(ROOT / "ai"), str(ROOT / "pipeline" / "src")]

PID = 100             # mã bệnh nhân trong bộ dữ liệu
INPUT = "dicom"       # "dicom": DICOM của 1 lần chụp (thư mục / zip) -> ai/dicom_to_nifti.py ; "nifti": file .nii.gz có sẵn
DICOM_SRC = None      # None: thư mục data/MRI_Data/01_MRI_Data/<PID> (máy local) hoặc hỏi upload zip (Colab)
RUN_SEGMENT = False   # True: chạy TotalSpineSeg thật (~18 s GPU, ~14 phút CPU)
SAVE_TO_DEMO = True   # True: ghi hình + JSON vào ai/demo/ (demo_<PID>.png, analysis_<PID>.json)
DEMO = ROOT / "ai" / "demo"
WORK = ROOT / "pipeline" / "outputs" / "cache" / f"demo_literature_{PID:04d}"              # đã gitignore
WORK.mkdir(parents=True, exist_ok=True)
# ảnh NIfTI đưa vào segment(): tạo từ DICOM vào WORK, hoặc file s02 đã ghi
IMAGE = WORK / f"{PID:04d}.nii.gz" if INPUT == "dicom" else ROOT / "pipeline" / "outputs" / "nifti" / "sag" / f"{PID:04d}.nii.gz"


def colab_upload(dest, what):
    """Colab trong trình duyệt: hộp chọn file -> ghi vào dest."""
    from google.colab import files as colab_files
    print(f"Chọn {what} để upload")
    up = colab_files.upload()
    assert len(up) == 1, "Upload đúng 1 file"
    name, data = next(iter(up.items()))
    stem = name.split(".")[0]
    # tên file và mask / JSON đều theo PID -> upload BN khác mà quên đổi PID là ghép nhầm mask
    assert not stem.isdigit() or int(stem) == PID, (
        f"File {name} là BN {int(stem)} nhưng PID = {PID}: sửa PID trong ô Cấu hình rồi chạy lại từ ô đó")
    dest.write_bytes(data)
    return dest

### B1. Ảnh đầu vào: DICOM hoặc NIfTI

In [ ]:
from common import MRI_DIR
from dicom_to_nifti import convert

if INPUT == "dicom":
    src = Path(DICOM_SRC) if DICOM_SRC else MRI_DIR / f"{PID:04d}"
    uploaded = None
    if not src.exists():
        assert IN_COLAB, f"Không có {src}: tải dữ liệu Mendeley vào data/ hoặc đặt DICOM_SRC"
        src = uploaded = colab_upload(WORK / "upload.zip", "file .zip DICOM của một lần chụp")
    try:
        info = convert(src, IMAGE)          # lỗi -> ConvertError(code, message), BE trả 400
    finally:
        if uploaded:
            uploaded.unlink()               # không giữ DICOM (header có thông tin cá nhân)
    print(f"DICOM: {info['n_series']} series -> chọn '{info['series_description']}' (số {info['series_number']}), "
          f"{info['n_slices']} lát, FOV {info['fov_mm']} mm, cách vùng axial {info['dist_ax_mm']} mm")
elif not IMAGE.exists():
    assert IN_COLAB, f"Không có {IMAGE}: chạy s02, hoặc đặt INPUT = \"dicom\""
    IMAGE.parent.mkdir(parents=True, exist_ok=True)
    colab_upload(IMAGE, f"file {IMAGE.name}")
im = sitk.ReadImage(str(IMAGE))
print("Ảnh:", IMAGE.relative_to(ROOT), im.GetSize(), "voxel", tuple(round(s, 2) for s in im.GetSpacing()), "mm")

### B2. Phân đoạn – `ai/segment.py`

In [ ]:
import shutil, subprocess
import segment as seg
from segment import segment

if RUN_SEGMENT:
    # TSS_CMD = CLI cạnh python của kernel, không có thì tên trần (tìm theo PATH, vd /usr/local/bin trên Colab)
    if not shutil.which(seg.TSS_CMD[0]):
        %pip install -q {PIP_C} "totalspineseg[nnunetv2]" "kornia<0.8"
    os.environ.setdefault("TOTALSPINESEG_DATA", str(ROOT / "pipeline" / ".tss_data"))
    r = segment(IMAGE, WORK / "output", timeout=3600)
    MASK = r.mask_path
    print(f"segment(): {r.duration_seconds} s")
else:
    found = lambda: next((p for p in [DEMO / f"{PID:04d}_seg.nii.gz",
                                      ROOT / "pipeline" / "outputs" / "tss" / "sag" / "step2_output" / f"{PID:04d}.nii.gz"]
                          if p.exists()), None)
    if not found():   # mask 515 ca của pipeline: GitHub Release, ~29 MB, kiểm tra SHA-256
        print("Chưa có mask -> tải mask 515 ca (s00_shared_data.py --fetch)")
        subprocess.run([sys.executable, "src/s00_shared_data.py", "--fetch"], cwd=ROOT / "pipeline", check=True)
    MASK = found()
    assert MASK, f"Không có mask của BN {PID} (không thuộc 515 ca?): đặt RUN_SEGMENT = True"
m_img = sitk.ReadImage(str(MASK))
# mask của bệnh nhân khác (vd quên đổi PID) có gốc tọa độ khác -> dừng thay vì phân tích sai
assert (m_img.GetSize() == im.GetSize() and np.allclose(m_img.GetOrigin(), im.GetOrigin(), atol=1e-2)
        and np.allclose(m_img.GetDirection(), im.GetDirection(), atol=1e-3)), (
    f"Mask {MASK.name} không cùng lưới với ảnh {IMAGE.name}: ảnh và mask không cùng bệnh nhân? Kiểm tra PID = {PID}")
labels = np.unique(sitk.GetArrayViewFromImage(m_img))
print("Mask:", MASK.relative_to(ROOT), "| nhãn:", [int(v) for v in labels if v])

### B3. Phân tích – `ai/analyze.py`

Đây chính là `result.analysis` mà BE trả cho FE.

Mỗi gợi ý có thêm dòng **`↳ theo y văn`** (trường `literature`): giá trị rơi vào mức nào theo ngưỡng y văn của s12.

Với BN 100, kết quả phải trùng [`analysis_0100.json`](analysis_0100.json). Điều này đúng với **cả hai** kiểu đầu vào, vì ảnh tạo từ DICOM trùng từng voxel với ảnh của pipeline. Nếu `SAVE_TO_DEMO = True`, JSON mới được ghi đè **sau** khi so.

In [ ]:
from analyze import analyze

a = analyze(IMAGE, MASK)
print("QC:", "đạt" if a["qc"]["ok"] else a["qc"]["warnings"], "| góc ưỡn:", a["lordosis_deg"], "độ")
for lv in a["levels"]:
    print(f"{lv['level']:7s} {len(lv['suggestions'])} gợi ý" + (f" (thiếu: {lv['missing']})" if lv["missing"] else ""))
    for s in lv["suggestions"]:
        print(f"   • {s['label_vi']} ({s['confidence']}, PPV {s['ppv']}): {s['measure_vi']} {s['value']} {s['rule']} {s['threshold']}"
              + (" (z)" if s["threshold_unit"] == "z" else "") + (f"\n       ↳ {s['literature']}" if s.get("literature") else ""))

ref = DEMO / f"analysis_with_literature_{PID:04d}.json"
if ref.exists():
    print("\nTrùng với", ref.name + ":", json.loads(json.dumps(a)) == json.loads(ref.read_text(encoding="utf-8")))
if SAVE_TO_DEMO:
    ref.write_text(json.dumps(a, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("Đã ghi", ref.relative_to(ROOT))

### B4. Vẽ hình

`analyze()` không trả tọa độ, nên đo lại bằng `measure_case(keep_pts=True)` của s09 để lấy:
- **lát giữa** `k_mid` – lát có nhiều voxel ống sống nhất;
- **tâm và trục** từng đĩa, để đặt nhãn tầng (màu đỏ nếu tầng có gợi ý).

Hàm `to_pixel()` của s09 đổi tọa độ mm sang pixel.

Dòng `↳ theo y văn: …` lấy từ trường `literature` của `analyze()`.

Hình được lưu vào `ai/demo/demo_<PID>.png` nếu `SAVE_TO_DEMO = True`, nếu không thì vào `WORK`.

Chữ "(z)" sau ngưỡng: quy tắc dùng ngưỡng z chung, nên số đo được so **sau khi chuẩn hóa z trong tầng**, không so trực tiếp giá trị mm.

In [ ]:
from measurements.s09_level_features import CANAL, measure_case, to_pixel

o = measure_case(MASK, IMAGE, keep_pts=True)
k, I, M, img = o["k_mid"], o["I"], o["M"], o["img"]
hits = {lv["level"]: lv["suggestions"] for lv in a["levels"]}


def color_of(lab):
    if lab in CANAL:
        return (0.35, 0.6, 0.95)        # tủy + ống sống
    if 41 <= lab <= 50:
        return (0.6, 0.45, 0.9)         # đốt L1-L5, L6, xương cùng
    if 91 <= lab <= 100:
        return (0.2, 0.75, 0.8)         # đĩa T12-L1 .. L5-S
    return (0.9, 0.3, 0.25) if lab < 60 else (0.95, 0.85, 0.2)   # đốt / đĩa cổ - ngực


overlay = np.zeros((*M[k].shape, 4))
for lab in np.unique(M[k]):
    if lab:
        overlay[M[k] == lab] = (*color_of(int(lab)), 0.45)

fig, (ax, tx) = plt.subplots(1, 2, figsize=(15, 7.5), gridspec_kw={"width_ratios": [1, 1.25]})
ax.imshow(I[k], cmap="gray", vmax=np.percentile(I[k], 99.5))
ax.imshow(overlay)
for lv, p in o["pts"].items():
    col = "red" if hits.get(lv) else "lime"
    a0 = to_pixel(img, k, *(p["c"] - p["e1"] * p["L"] * 1.3))   # -e1 = phía trước đĩa
    a1 = to_pixel(img, k, *p["c"])
    ax.plot([a0[0], a1[0]], [a0[1], a1[1]], color=col, lw=1)
    ax.text(a0[0] - 3, a0[1], lv, color=col, fontsize=8, ha="right", va="center")
ax.set_title(f"BN {PID:04d} ({INPUT.upper()}) - lát giữa {k}: ảnh T2 + mask (đỏ = có gợi ý)")
ax.axis("off")

lines = []
if not a["qc"]["ok"]:
    lines += ["QC không đạt - không gợi ý:"] + [f"  ! {w['message']}" for w in a["qc"]["warnings"]] + [""]
for lv in a["levels"]:
    lines.append(lv["level"] + (f"  (thiếu: {lv['missing']})" if lv["missing"] else ""))
    lines += [f"   • {s['label_vi']} ({s['confidence']}, PPV {s['ppv']})  {s['measure_vi']} {s['value']} "
              f"{s['rule']} {s['threshold']}" + (" (z)" if s["threshold_unit"] == "z" else "")
              + (f"\n        ↳ {s['literature']}" if s.get("literature") else "")
              for s in lv["suggestions"]] or ["   – không có gợi ý"]
lines += ["", a["disclaimer"]]
tx.text(0, 1, "\n".join(lines), va="top", fontsize=9.5, linespacing=1.5)
tx.set_title("analyze() - gợi ý theo tầng")
tx.axis("off")
fig.tight_layout()
out = (DEMO if SAVE_TO_DEMO else WORK) / f"demo_{PID:04d}.png"
fig.savefig(out, dpi=110)
print("Đã lưu", out.relative_to(ROOT))
plt.show()